Import the reuqired librearies

In [8]:
import os
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader

Tranforms the images 

In [9]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(10),

    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])


Fetch the Datasets

In [10]:


train_dir = "C:/Users/Dell/Documents/Internship_project/Brain Tumor MRI Image Classification/train/train"
valid_dir = "C:/Users/Dell/Documents/Internship_project/Brain Tumor MRI Image Classification/valid/valid"
test_dir = "C:/Users/Dell/Documents/Internship_project/Brain Tumor MRI Image Classification/test/test"

train_datasets = datasets.ImageFolder(train_dir, transform=train_transform)
valid_datasets = datasets.ImageFolder(valid_dir, transform=val_transform)
test_datasets = datasets.ImageFolder(test_dir, transform=val_transform)


Create the DataLoader

In [11]:
train_loader = DataLoader(train_datasets, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(valid_datasets, batch_size=32, shuffle=False)
test_loader = DataLoader(test_datasets, batch_size=32, shuffle=False)


In [12]:
print(len(train_datasets.classes))

4


Model Define

In [5]:
import torch.nn as nn
import torch.nn.functional as F
import torch

class BrainTumorCNN(nn.Module):
    def __init__(self):
        super(BrainTumorCNN, self).__init__()
        self.conv1 = nn.Conv2d(3, 16, kernel_size=3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, kernel_size=3, padding=1)
        self.conv3 = nn.Conv2d(32, 64, kernel_size=3, padding=1)
        self.dropout = nn.Dropout(0.5)

        # ✅ Use dynamic flatten size calculation
        with torch.no_grad():
            dummy_input = torch.zeros(1, 3, 224, 224)
            x = self.pool(F.relu(self.conv1(dummy_input)))
            x = self.pool(F.relu(self.conv2(x)))
            x = self.pool(F.relu(self.conv3(x)))
            self.flatten_size = x.view(1, -1).shape[1]

        self.fc1 = nn.Linear(self.flatten_size, 256)
        self.fc2 = nn.Linear(256, 4)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = self.pool(F.relu(self.conv3(x)))
        x = torch.flatten(x, 1)  # ✅ No view
        x = self.dropout(F.relu(self.fc1(x)))
        x = self.fc2(x)
        return x
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# Class weights (inverse of frequency)
weights = torch.tensor([1/564, 1/358, 1/335, 1/438])
weights = weights / weights.sum()  # Normalize
weights = weights.to(device)

# Replace this:
# criterion = nn.CrossEntropyLoss()

# With this (restore your weighted loss):
weights = torch.tensor([1/564, 1/358, 1/335, 1/438])
weights = weights / weights.sum()
weights = weights.to(device)

criterion = nn.CrossEntropyLoss(weight=weights)


In [14]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BrainTumorCNN().to(device)


In [15]:
criterion = nn.CrossEntropyLoss()                     # Suitable for multi-class classification
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)


In [23]:
num_epochs = 4

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        print(f"Epoch {epoch+1}/{num_epochs}, Loss: {running_loss/len(train_loader):.4f}")



Epoch 1/4, Loss: 0.0113
Epoch 1/4, Loss: 0.0248
Epoch 1/4, Loss: 0.0355
Epoch 1/4, Loss: 0.0456
Epoch 1/4, Loss: 0.0593
Epoch 1/4, Loss: 0.0675
Epoch 1/4, Loss: 0.0772
Epoch 1/4, Loss: 0.0917
Epoch 1/4, Loss: 0.1040
Epoch 1/4, Loss: 0.1117
Epoch 1/4, Loss: 0.1224
Epoch 1/4, Loss: 0.1335
Epoch 1/4, Loss: 0.1453
Epoch 1/4, Loss: 0.1550
Epoch 1/4, Loss: 0.1631
Epoch 1/4, Loss: 0.1735
Epoch 1/4, Loss: 0.1878
Epoch 1/4, Loss: 0.1995
Epoch 1/4, Loss: 0.2072
Epoch 1/4, Loss: 0.2125
Epoch 1/4, Loss: 0.2224
Epoch 1/4, Loss: 0.2324
Epoch 1/4, Loss: 0.2404
Epoch 1/4, Loss: 0.2569
Epoch 1/4, Loss: 0.2663
Epoch 1/4, Loss: 0.2746
Epoch 1/4, Loss: 0.2840
Epoch 1/4, Loss: 0.2921
Epoch 1/4, Loss: 0.3020
Epoch 1/4, Loss: 0.3105
Epoch 1/4, Loss: 0.3201
Epoch 1/4, Loss: 0.3294
Epoch 1/4, Loss: 0.3385
Epoch 1/4, Loss: 0.3460
Epoch 1/4, Loss: 0.3533
Epoch 1/4, Loss: 0.3599
Epoch 1/4, Loss: 0.3706
Epoch 1/4, Loss: 0.3825
Epoch 1/4, Loss: 0.4009
Epoch 1/4, Loss: 0.4115
Epoch 1/4, Loss: 0.4171
Epoch 1/4, Loss:

Model Accuracy

In [25]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in val_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Validation Accuracy: {accuracy:.2f}%")


Validation Accuracy: 86.85%


In [22]:
from PIL import Image

# Load and preprocess test image
image_path = "C:/Users/Dell/Documents/Internship_project/Brain Tumor MRI Image Classification/test/test/glioma/Tr-gl_0018_jpg.rf.7a670766b8083a1b516a49e241a636bc.jpg"  # give path to one MRI image
image = Image.open(image_path).convert("RGB")

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.5], [0.5])
])

img_tensor = transform(image).unsqueeze(0).to(device)

# Predict
model.eval()
with torch.no_grad():
    output = model(img_tensor)
    _, predicted = torch.max(output, 1)

class_names = train_datasets.classes  # ['glioma', 'meningioma', 'no_tumor', 'pituitary']
print("Predicted class:", class_names[predicted.item()])


Predicted class: glioma


In [19]:
torch.save(model.state_dict(), 'brain_tumor_model.pth')


In [2]:
from collections import Counter
import os

dataset_path = "C:/Users/Dell/Documents/Internship_project/Brain Tumor MRI Image Classification/train/train"
classes = os.listdir(dataset_path)
for cls in classes:
    count = len(os.listdir(os.path.join(dataset_path, cls)))
    print(f"{cls}: {count}")


glioma: 564
meningioma: 358
no_tumor: 335
pituitary: 438


NotADirectoryError: [WinError 267] The directory name is invalid: 'C:/Users/Dell/Documents/Internship_project/Brain Tumor MRI Image Classification/train/train\\_classes.csv'